In [7]:
### Create files of ssDNA and dsDNA viruses
import polars as pl

# !wget https://portal.nersc.gov/cfs/m342/UHGV/metadata/uhgv_metadata.tsv
# !wget https://portal.nersc.gov/cfs/m342/UHGV/metadata/votus_metadata.tsv

uhgv_ictv_taxonomy = (
    pl.read_csv('../../figure_1/uhgv_metadata.tsv', separator='\t', columns=['uhgv_genome', 'uhgv_votu'])
        .join(
            pl.read_csv('../votus_metadata.tsv', separator='\t', columns=['uhgv_votu', 'ictv_taxonomy', 'genome_length']),
            on='uhgv_votu', how='left'
        )
)

votu_info = (
    pl.read_csv('../../figure_2/vclust/uhvdb_vclust_votu_reps_final.tsv', separator='\t', new_columns=['seq_id'])
        .join(
            pl.read_csv('../../figure_1/viruses.csvtk_concat.tsv', separator='\t', columns=['seq_name', 'taxonomy', 'contig_length', 'proviral_length']),
            left_on='seq_id', right_on='seq_name', how='left'
        )
        .join(
            pl.read_csv('../../figure_1/uhvdb_final_metadata.tsv', separator='\t'),
            left_on='seq_id', right_on='seq_name', how='left'
        )
        .join(
            uhgv_ictv_taxonomy, left_on='seq_id', right_on='uhgv_genome', how='left'
        )
        .with_columns([
            pl.col('taxonomy').fill_null(pl.col('ictv_taxonomy')),
            pl.when(pl.col('contig_length').is_not_null())
                .then(pl.col('contig_length'))
                .when(pl.col('proviral_length').is_not_null())
                .then(pl.col('proviral_length'))
                .otherwise(pl.col('genome_length')).alias('length').cast(pl.Float64)
        ])
        .unique('seq_id')
)

def extract_realm(value):
    if value is None:
        return ""
    parts = value.split(';')
    if len(parts) > 1:
        if parts[1].strip() == "Anelloviridae":
            return "Monodnaviria"
        if parts[1].strip() == "Naldaviricetes":
            return "Duplodnaviria"
        else:
            return parts[1].strip()
    return ""

print(
    votu_info
        .with_columns([
            pl.col('taxonomy').map_elements(extract_realm, return_dtype=pl.String).alias('ictv_realm'),
        ])
        .group_by('ictv_realm')
        .len()
        .sort('len', descending=True)
)

# # create a list of dsDNA viruses
# (
#     votu_info
#         .with_columns([
#             pl.col('taxonomy').map_elements(extract_realm, return_dtype=pl.String).alias('ictv_realm'),
#         ])
#         .filter(
#             (pl.col('ictv_realm') == 'Duplodnaviria')
#             # (pl.col('ictv_realm') == 'Varidnaviria')
#         )[['seq_id']]
#         .write_csv('../dsDNA_viruses.tsv', separator='\t', include_header=False)
# )

# #create a list of ssDNA viruses
# (
#     votu_info
#         .with_columns([
#             pl.col('taxonomy').map_elements(extract_realm, return_dtype=pl.String).alias('ictv_realm'),
#         ])
#         .filter(
#             (pl.col('ictv_realm') == 'Monodnaviria')
#         )[['seq_id']]
#         .write_csv('../ssDNA_viruses.tsv', separator='\t', include_header=False)
# )

shape: (5, 2)
┌───────────────┬────────┐
│ ictv_realm    ┆ len    │
│ ---           ┆ ---    │
│ str           ┆ u32    │
╞═══════════════╪════════╡
│ Duplodnaviria ┆ 176413 │
│ Monodnaviria  ┆ 21910  │
│ Varidnaviria  ┆ 2584   │
│               ┆ 528    │
│ Riboviria     ┆ 510    │
└───────────────┴────────┘


In [ ]:
# ### compile C++ code to convert similarity TSV to PHYLIP distance matrix
# !g++ -O2 -std=c++17 ../tsv_to_phylip.cpp -o ../tsv_to_phylip -lz

In [ ]:
# ### Create norm score TSV (pruned at family level) for dsDNA viruses only 
# import polars as pl

# dsdna_viruses = set(
#     pl.read_csv('../dsDNA_viruses.tsv', separator='\t', has_header=False, new_columns=['seq_id'])['seq_id']
# )

# df = (
#     pl.scan_csv('uhvdb_all.normscores.tsv.gz', separator='\t')
#         .filter(
#             (pl.col('query').is_in(dsdna_viruses)) &
#             (pl.col('reference').is_in(dsdna_viruses))
#         )
#         .sink_csv('dsdna_normscores.tsv', separator='\t', engine='streaming')
# )

In [ ]:
# ### convert distance to PHYLIP format
# !../tsv_to_phylip \
#     dsdna_normscores.tsv \
#     dsdna_normscores.phylip \
#     --similarity

In [ ]:
# ### Create phylogeny using DIPPER
# !dipper \
#     -i d \
#     -o t \
#     -I dsdna_normscores.phylip \
#     -O dsdna_normscores.nwk

In [9]:
### create annotation file for dsDNA viruses
import polars as pl

# identify ssdna families
dsdna_viruses = set(pl.read_csv('../dsDNA_viruses.tsv', separator='\t', has_header=False, new_columns=['seq_id'])['seq_id'])
family_clusters = pl.read_csv('../../figure_2/protein_similarity/uhvdb_family_clusters.tsv', separator='\t')

### Family cluster IDs
family_clusters = pl.read_csv('../../figure_2/protein_similarity/uhvdb_family_clusters.tsv', separator='\t')
# # 1. calculate median length amd size each cluster
# dsdna_largest_families = set(
#     family_clusters
#         .filter(
#             (pl.col('contig_id').is_in(dsdna_viruses))
#         )
#         .group_by('cluster_id')
#         .len()
#         .sort('len', descending=True).head(10)['cluster_id']
# )
# print(dsdna_largest_families)

family_clusters = (
    family_clusters
        .filter(
            (pl.col('contig_id').is_in(dsdna_viruses))
        )
        .filter(pl.col('cluster_id').is_in([2, 3, 5, 6, 8, 9, 10, 11, 12, 13]))
        .with_columns([(pl.lit('cluster_') + pl.col('cluster_id').cast(pl.Utf8)).alias('cluster_id')])
        .rename({'contig_id':'seq_name'})
        [['seq_name', 'cluster_id']]
)


## ICTV Class ###
def extract_class(value):
    if value is None:
        return ""
    parts = value.split(';')
    if len(parts) > 1:
        for part in parts:
            if part.endswith('viricetes'):
                return part.strip()
            if part == 'Anelloviridae':
                return 'Cardeaviricetes'
            if part == 'Bamfordvirae':
                return 'Bamfordvirae-Class'
    return ""

ictv_class_info = (
    votu_info
        .filter(pl.col('seq_id').is_in(dsdna_viruses))
        .with_columns([
            pl.col('taxonomy').map_elements(extract_class, return_dtype=pl.String).alias('ictv_class')
        ])
        [['seq_id', 'ictv_class']]
)
print(ictv_class_info.group_by('ictv_class').len().sort('len', descending=True))

ictv_classes = (
    pl.read_csv('../../figure_2/protein_similarity/uhvdb_family_clusters.tsv', separator='\t')
        .join(ictv_class_info, left_on='contig_id', right_on='seq_id', how='left')
        .filter(pl.col('contig_id').is_in(dsdna_viruses))
        .filter(pl.col('ictv_class').is_in(['Caudoviricetes', 'Herviviricetes']))
        .rename({'contig_id':'seq_name'})
        [['seq_name', 'ictv_class']]
)

### Identify if sequence is new or from prior database
db_type = (
    pl.read_csv('../../figure_1/uhvdb_final_metadata.tsv', separator='\t')[['seq_name', 'db_type']]
        .filter(pl.col('seq_name').is_in(dsdna_viruses))
        [['seq_name', 'db_type']]
)


body_site = (
    pl.read_csv('../../figure_1/uhvdb_final_metadata.tsv', separator='\t')[['seq_name', 'body_site']]
        .filter(pl.col('seq_name').is_in(dsdna_viruses))
        .with_columns([
            pl.when(pl.col('body_site') == 'Oral').then(pl.lit('Airways'))
                .when(pl.col('body_site') == 'Gut').then(pl.lit('Gut'))
                .when(pl.col('body_site') == 'Skin').then(pl.lit('Skin'))
                .when(pl.col('body_site') == 'Urogenital').then(pl.lit('Urogenital'))
                .otherwise(pl.lit('Other')).alias('body_site'),
        ])[['seq_name', 'body_site']]
)


### Annotate with completeness 
completeness = (
    pl.read_csv('../../figure_1/uhvdb_final_metadata.tsv', separator='\t')[['seq_name', 'checkv_quality']]
        .filter(pl.col('seq_name').is_in(dsdna_viruses))
        .with_columns([
            pl.when(pl.col('checkv_quality') == 'Complete').then(pl.lit('Complete'))
                .when(pl.col('checkv_quality') == 'High-quality').then(pl.lit('High-quality')).alias('checkv_quality'),
        ])[['seq_name', 'checkv_quality']]
)

### combine all annotations
all_annotations = (
    family_clusters
        .join(ictv_classes, on='seq_name', how='full', coalesce=True)
        .join(db_type, on='seq_name', how='full', coalesce=True)
        .join(body_site, on='seq_name', how='full', coalesce=True)
        .join(completeness, on='seq_name', how='full', coalesce=True)
        .select([
            'seq_name', 'cluster_id', 'ictv_class', 'db_type', 'body_site', 'checkv_quality'
        ])
)
all_annotations.write_csv('../dsdna_annotations/all_dsdna_annotations.tsv', separator='\t')

shape: (4, 2)
┌────────────────┬────────┐
│ ictv_class     ┆ len    │
│ ---            ┆ ---    │
│ str            ┆ u32    │
╞════════════════╪════════╡
│ Caudoviricetes ┆ 176378 │
│                ┆ 19     │
│ Herviviricetes ┆ 15     │
│ Naldaviricetes ┆ 1      │
└────────────────┴────────┘


In [11]:
!newick_to_taxonium \
    --input dsdna_normscores.nwk \
    --output dsdna_taxonium.json \
    --metadata ../dsdna_annotations/all_dsdna_annotations.tsv \
    --columns cluster_id,ictv_class,db_type,body_site,checkv_quality \
    --config_json config.json \
    --title "UHVDB dsDNA phylogeny" \
    --key_column seq_name

Loading metadata file..
Metadata loaded
/mmfs1/gscratch/pedslabs_hoffman/carsonjm/micromamba_envs/envs/lgonsa/lib/python3.12/site-packages/taxoniumtools/newick_to_taxonium.py:48: ResourceWarning: unclosed file <_io.TextIOWrapper name='config.json' mode='r' encoding='UTF-8'>
  config = json.load(open(config_file))
Ladderizing tree..
Ladderizing done
Setting x coordinates |████████████████████████████████████████| 352823 in 1.5s 8 in 1▶▶▶▶▶▶▶                         | ▂▄▆ 48280 in 0s▃▅▇ 95805 in 0sin 1▅▇▇ 189139 in 1284707 in 1
Normalising x coordinates |████████████████████████████████████████| 352823/352853052/3 102129/ ▄▆█ 132120/ 280518/ 312239/
Setting terminal y coordinates |████████████████████████████████████████| 176412 93|   ▶▶▶▶▶▶▶▶▶▶▶▶▶▶                       | ▄▆█ 13
Setting internal y coordinates |████████████████████████████████████████| 17641112 ▅▇▇ 16
Sorting on y |████████████████████████████████████████| 352823 in 0.5s (670385.7 62822 in 0s (386191.(462919 229728 in 0s